#  Artificial Intelligence Systems Assignment 1


**Assignment due: Mar 29, 2026, 23:59**.

Automatic differentiation is the foundation technique of training a machine learning model.
In this assignment, you will implement a simple prototype automatic differentiation system.

* You should work on this assignment **individually** -- it is not a team assignment.
* This assignment does not require GPU. You can do the assignment on your laptop/desktop, or CPU server in Qizhi.
* This assignment is pure Python. No C++ is needed in this assignment.
* Please check out the end of this notebook for the assignment submission requirement.
* Please do not share your solution on publicly available websites (e.g., GitHub).
* Please write **clean** and **readable** code. Submissions with excessively poor code quality may be subject to point deductions.
* Please submit your **Code Agent Dialogue**.
* **About testing and grading.** The assignment will be automatically graded. The test cases include both public tests that we provide under `tests/`, as well as some private tests (which will not be disclosed). You can submit multiple times, and the time stamp of that submission will be used in determining any late penalties. The scores you get in each task is proportional to the number of total test cases you pass. We also encourage you to create your own test cases, which helps you confirm the correctness of your code.


## Automatic Differentiation Framework (100 pt)

In part 1, you will implement the reverse mode automatic differentiation algorithm.

The auto diff algorithm in this assignment works on a **computational graph**.
A computational graph describes the process of computation of an expression.
For example, given $x_1$, $x_2$, the expression $y = x_1 \times x_2 + x_1$ has the following computational graph:

<img src="./figure/computational_graph.jpg" alt="figure/computational_graph.jpg" width="60%"/>

Let's first walk you through the basic concepts and data structures in the framework.
A computational graph consists of **nodes**, where each node denotes an intermediate step of computation during computing the entire expression.
Every node is composed of the three parts (`auto_diff.py` line 6):

- an **operation** (field `op`), which defines the operation that the node computes.
- a list of **input nodes** (field `inputs`), which indicates the input source of the computation.
- optionally, additional "**attributes**" (field `attrs`). The attributes that a node has depends on the op of the node. We will explain the attributes later in this part.

We can define an input node of a computational graph with `ad.Variable`. For example, the input variable nodes $x_1$ and $x_2$ can be defined as

```python
import auto_diff as ad

x1 = ad.Variable(name="x1")
x2 = ad.Variable(name="x2")
```

In `auto_diff.py` (line 81), you can see that the essence of `ad.Variable` is to construct a node
with op `placeholder` and the given name. The input nodes have empty `inputs` and `attrs`:
```python
class Variable(Node):
    def __init__(self, name: str) -> None:
        super().__init__(inputs=[], op=placeholder, name=name)
```

Here, the `placeholder` defines the computation of a input variable node, which is "doing nothing."
Besides `placeholder`, we have other ops defined in `auto_diff.py`. For example,

- op `add` defines the addition of two nodes,
- op `matmul` defines the matrix multiplication of two nodes.

Notably, these ops are globally defined for only once, and the `op` field of every node is such
a globally defined op.

Now, back to our example of $y = x_1 \times x_2 + x_1$.
Now that we have `x1` and `x2` as two input variable nodes, we can define the rest of the computational graph
with a one-line Python code:
```python
y = x1 * x2 + x1
```

This line first constructs a node with op `mul` (multiplication) and `x1`, `x2` as `inputs`,
and then constructs a node with op `add` which takes the previous multiplication node and `x1` as `inputs`.
As a result, our computational graph contains four nodes in the end.

It worths noting that a computational graph (e.g., the four nodes we defined) **does not** carry concrete values of nodes.
The style of this assignment is consistent with the TensorFlow v1 style, introduced in the lecture.
This is different from frameworks like PyTorch, where the values of input tensors will be given in the beginning,
and the values of intermediate tensors are eagerly computed along the way when those tensors are defined.
In our computational graph, to compute the value of output `y` given values of input `x1`, `x2`,
we provide the `Evaluator` class (`auto_diff.py` line 373).

Here is an example of how `Evaluator` works. The constructor of `Evaluator` takes a list of nodes to evaluate.
By writing
```python
evaluator = ad.Evaluator(eval_nodes=[y])
```
it means that we construct an `Evaluator` instance which aims to compute the value of `y`.
Then we provide the values (assuming all `numpy.ndarray` in this assignment) of the input tensors through the main interface `Evaluator.run` (which you need to implement):
```python
import numpy as np

x1_value = np.array(2)
x2_value = np.array(3)
y_value = evaluator.run(input_dict={x1: x1_value, x2: x2_value})
```

At a high level, here the `run` method consumes the input values via a dictionary `Dict[Node, numpy.ndarray]`,
computes the value of node `y` internally, and returns the result.
Given `2 * 3 + 2 = 8`, the returned `y_value` should be `np.ndarray(8)` eventually (of course, it will not return the correct value before your finish implementing):
```python
np.testing.assert_allclose(y_value, np.array(8))
```

`Evaluator.run` method effectively computes the forward computation of nodes, and we can go ahead to talk about the backward.
As you learned in the lecture, in order to compute the output gradient with regard to each input node in a computational graph,
we can extend the forward graph with the additional backward part.
Once we have the forward and backward graph together, by given input node values,
we can use `Evaluator` to compute the output value, loss value, and the gradient values of each input nodes altogether with a single-time `Evaluator.run`.

The function `gradients(output_node: Node, nodes: List[Node]) -> List[Node]` in `auto_diff.py` is
the function you need to implement to construct the backward graph.
This function takes an output node (usually the node of the loss function in machine learning), whose gradient is treated as 1,
takes the list of nodes to compute gradients for,
and returns the gradient nodes with regard to each node in the input list.

Back to our example, after implementing `gradients`, you can run
```python
x1_grad, x2_grad = ad.gradients(output_node=y, node=[x1, x2])
```
to get the gradients of $y$ regarding $x_1$ and $x_2$ respectively.
And you can construct `Evaluator` on nodes `y`, `x1_grad` and `x2_grad`, and use `Evaluator.run`
to compute the output value and input gradients.

Finally, before leaving the assignment to you, we introduce how op works.
As you can find in `auto_diff.py`, each op defines three methods:

- `__call__(self, **kwargs) -> Node`, which takes in the input nodes (and attributes), constructs a new node with this op, and returns the constructed node.
- `compute(self, node: Node, input_values: List[np.ndarray]) -> np.ndarray`, which takes the node to compute with its input values, and returns the computed value of the given node.
- `gradient(self, node: Node, output_grad: Node) -> List[Node]`, which takes a node and the gradient node of this node, and returns the nodes of the input partial adjoints (one for each input node).

In general, the `Op.compute` method computes the value of a single node with given node inputs, and `Evaluator.run` function computes the value of a graph output with given graph inputs.
`Op.gradient` method constructs the backward computational graph for a single node, and `gradients` function constructs the backward graph for a graph.
That being said, your implementation of `Evaluator.run` should effectively make use of the `compute` method of op,
and likewise, the `gradients` function implementation should leverage the `gradient` method defined in op.

### Your tasks

**Task 1 (20 pt).** Implement the `compute` method for all ops in `auto_diff.py`. We provide the examples of `AddOp` and `AddByConstOp`, and you need to implement the rest.
For this assignment, you can assume that the inputs of addition/multiplication/division have the same shape.

We provide sample tests in `tests/test_auto_diff_node_forward.py`.
You can test your task 1 implementation by running

In [ ]:
!python3 -m pytest -l -v tests/test_auto_diff_node_forward.py

**Task 2 (25 pt).** Implement the `Executor.run` method in `auto_diff.py`.
You may want to get the [topological sort](https://en.wikipedia.org/wiki/Topological_sorting) of the computational graph
in order to compute the output value.

We provide sample tests in `tests/test_auto_diff_graph_forward.py`.
You can test your task 2 implementation by running

In [ ]:
!python3 -m pytest -l -v tests/test_auto_diff_graph_forward.py

**Task 3 (25 pt).** Implement the `gradient` method for all ops in `auto_diff.py`. We provide the examples of `AddOp` and `AddByConstOp`, and you need to implement the rest.

We provide sample tests in `tests/test_auto_diff_node_backward.py`.
You can test your task 3 implementation by running

In [ ]:
!python3 -m pytest -l -v tests/test_auto_diff_node_backward.py

**Task 4 (30 pt).** Implement `gradients` function in `auto_diff.py`.
You may also find topological sort helpful in the implementation.

We provide sample tests in `tests/test_auto_diff_graph_backward.py`.
You can test your task 4 implementation by running

In [ ]:
!python3 -m pytest -l -v tests/test_auto_diff_graph_backward.py

### A few notes

1. **Zero-rank arrays in NumPy.** As mentioned earlier, all values are assumed to have type `numpy.ndarray` throughout this assignment. One thing you may find interesting about NumPy is, if we add two zero-rank arrays together (e.g., `np.array(1) + np.array(2)`), it results in a scalar value, rather than a zero-rank array:
```
>>> x = np.array(1)
>>> y = np.array(2)
>>> type(x), type(y), x.ndim, y.ndim
(<class 'numpy.ndarray'>, <class 'numpy.ndarray'>, 0, 0)
>>> z = x + y
>>> z, type(z)
(3, <class 'numpy.int64'>)
```
This means that, if you want to have a rigorous implementation of your assignment, you need to check the result type
at the end of `compute` methods, and wraps any scalar values back to `numpy.ndarray`.
However, for simplicity, we do not requrire you to do this, and it is completely up to you:
there will be no test for this behavior, and you won't get fewer credits because of not doing this.
Python by default also does not have eager type checking to throw error when you do not handle the scalars.

2. **`Node.attrs`.** In the reference implementation of `AddByConstOp` in `auto_diff.py`, you will find that the `attrs` field is used to store the constant operand of the addition in the returned node. In general, the `attrs` field of a node stores all the **constants** that are known when constructing the computational graph: for the case of `AddByConstOp`, the constant operand is stored as a node attribute. While for general cases, an attribute does not have to be a node operand. You can see that in `MatMulOp` in `auto_diff.py`, we store the boolean flags denoting whether to transpose the input matrices as attributes. In the next part of this assignment, you may implement op like `SumOp`, and find it useful to store the axis being reduced as a node attribute.

3. **Minimality of `gradients`.** The `gradients` function constructs the backward graph and returns the gradient nodes with regards to required nodes. One interesting note here is the minimality of the constructed backward graph. For example, for a graph of `y = x1 * x2 + x1`, if we are only interested in the gradient of `x1 * x2`, a minimal backward graph only contains the gradient node of `x1 * x2`, which means it is not necessary to construct the gradient nodes for `x1` and `x2`. In this assignment, we **do not** require you to construct the minimal backward graph, but it would be a good mental exercise to think about the possible pros/cons of constructing minimal backward graphs.

## Part 3. Learning Journal (0 pt)

Please submit a brief journal about your experience with this assignment.  
Include your reflections in a file named `journal.txt` and submit it with your code.

- What are the key lessons or skills you learned from this assignment?
- What specific tasks did you complete? 
- What difficulties did you encounter? How did you solve them?

This journal is required but not graded. 

## Part 4. Assignment Feedback (0 pt)

This is the first time we offer this course, and we appreciate any assignment feedback from you.
You can leave your feedback (if any) in `feedback.txt`, and submit it together with the source code.
Possible choices can be:

- How difficult do you think this assignment is?
- How much time does the assignment take? Which task takes the most time?
- Which part of the assignment do you feel hard to understand?
- And any other things you would like to share.

Your feedback will be very useful in helping us improve the assignment quality
for next years.


## How to Submit Your Code

In the home directory for the assignment, execute the command

In [ ]:
!zip handin.zip auto_diff.py tests/test_customized_cases.py journal.txt feedback.txt

This will create a zip file with `auto_diff.py`, `tests/test_customized_cases.py`, `journal.txt` and `feedback.txt`.
You can check the contents of `handin.zip` to make sure it contains all the needed files:

In [ ]:
!zipinfo -1 handin.zip

It is expected to list the four files:
```
auto_diff.py
tests/test_customized_cases.py
journal.txt
feedback.txt
```

The submission link will be given later.

The assignment will be automatically graded. The test cases include both public tests that we provide under `tests/`,
as well as some private tests (which will not be disclosed).
You can submit multiple times, and the time stamp of that submission will be used in determining any late penalties.
Please make sure that your submitted `auto_diff.py` are placed at the root level of the zip file (i.e., they are not in any sub-folder),
or **otherwise the autograder may not process your submission properly**.

**Any attempt to manipulate or compromise the integrity of the autograder will result in severe penalties.**
